# Fed-MOPSD: full-parameter federated on-policy self-distillation

This notebook implements one reproducible **FedAvg-style Fed-MOPSD** experiment with three
domain clients. Every client receives the same global Qwen3-4B-Base student, keeps its
`(question, reference answer)` pairs local, generates student on-policy rollouts, and evaluates
those rollouts with a frozen copy of the **initial** Qwen3-4B-Base conditioned on the privileged
reference answer. Only the local student receives gradients. Every complete local Student is
saved independently before the three client checkpoints are averaged into the next global Student.

Default data assignment (exactly 3,000 usable records per client):

- Math: `siyanzhao/Openthoughts_math_30k_opsd`
- Instruction following: `allenai/tulu-3-sft-personas-instruction-following`
- Code: `microsoft/rStar-Coder`, configuration `seed_sft`, retaining only
  `verified == True and is_passed == True`

The final aggregated student is evaluated with the same five benchmark protocols used in
`1.eval_student.ipynb`: AIME25, AIME26, IFBench, IFEval, and LiveCodeBench release_v6. All
generation paths use `MAX_NEW_TOKENS = 1024`. Training and evaluation are resumable.

**Resource note.** This is full-parameter training, not LoRA. Physical GPU 1 is exposed as the
only device, so both the trainable Student and frozen Teacher use logical `cuda:0`. Adafactor, BF16, gradient
checkpointing, micro-batch 1, and gradient accumulation reduce memory, but 3 × 3,000 on-policy
generations plus full-vocabulary distillation is still a long experiment. Each client checkpoint
is roughly one BF16 model, so reserve substantial disk space.

The conservative full-parameter learning rates are Math `1e-6`, IF `5e-7`, and Code `8e-7`.
IF uses the smallest value because exact formatting constraints are especially vulnerable to
destructive drift; Math and Code receive slightly larger updates for long-form reasoning.

This notebook is a single-machine simulation of three logical clients. It preserves the algorithmic
data boundary in the training loop, but it is not a network deployment and does not implement secure
aggregation or differential privacy.


In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# The mirror/Xet combination can return CAS 401 for public model shards; force regular HTTP.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "1800")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
print("HF mirror:", os.environ["HF_ENDPOINT"], flush=True)
print("HF Xet disabled:", os.environ["HF_HUB_DISABLE_XET"], flush=True)
print("Physical CUDA devices exposed to this kernel:", os.environ["CUDA_VISIBLE_DEVICES"], flush=True)


In [ ]:
# Papermill parameters
MODEL_NAME = "Qwen/Qwen3-4B-Base"  # Model identity used in experiment records.
MODEL_SOURCE = "/root/models/Qwen3-4B-Base"  # ModelScope local snapshot used for all loads.
MODEL_REVISION = "modelscope-master"
OUTPUT_DIR = "results/fed_mopsd_qwen3_4b_base"
CLIENT_DATA_DIR = "results/fed_mopsd_qwen3_4b_base/selected_client_data"
NLTK_DATA_DIR = "~/nltk_data"
SEED = 42

CLIENT_SAMPLES = 2000
FED_ROUNDS = 1
LOCAL_EPOCHS = 1
DOMAIN_WEIGHTING = "equal"  # Equal and sample-weighted coincide because every Client uses CLIENT_SAMPLES records.

LR_MATH = 1.0e-6
LR_IF = 5.0e-6
LR_CODE = 5.0e-6
WEIGHT_DECAY = 0.0
MICRO_BATCH_SIZE = 1  # One prompt is processed at a time; its rollouts are scored sequentially.
ROLLOUTS_PER_PROMPT = 1  # Number of independently sampled on-policy responses for each question.
GRADIENT_ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = (
    MICRO_BATCH_SIZE * ROLLOUTS_PER_PROMPT * GRADIENT_ACCUMULATION_STEPS
)  # rollout sequences contributing to each optimizer update
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.03

MAX_NEW_TOKENS = 1024
MAX_TRAIN_SEQ_LENGTH = 4096
MAX_QUESTION_TOKENS = 1400
MAX_REFERENCE_TOKENS = 2000
TRAIN_TEMPERATURE = 1.0
TRAIN_TOP_P = 1.0
TRAIN_TOP_K = 0
DISTILL_TEMPERATURE = 1.0
KL_TOKEN_CHUNK = 16

CHECKPOINT_EVERY_OPTIMIZER_STEPS = 50
KEEP_LAST_CHECKPOINTS = 1
LOSS_CURVE_SMOOTHING_WINDOW = 10
MODEL_SHARD_SIZE = "2GB"
FEDAVG_SHARD_SIZE_BYTES = 2_000_000_000
MIN_FREE_DISK_GB_WARNING = 45
RESUME = True
INSTALL_DEPS = False

# Evaluation parameters: intentionally identical in meaning to 1.eval_student.ipynb.
LIMIT_AIME25 = None
LIMIT_AIME26 = None
LIMIT_IFBENCH = None
LIMIT_IFEVAL = None
LIMIT_LCB = None
EVAL_MAX_SEQ_LENGTH = 32768
TEMPERATURE = 0.6
TOP_P = 0.95
TOP_K = 20
MIN_P = 0.0
RUN_MATH = True
RUN_IF = True
RUN_LCB = True
RUN_LCB_EVALUATION = True
LCB_EVAL_ONLY = False
LCB_RELEASE_VERSION = "release_v6"
LCB_DATASET_REVISION = "d44be6b144381afa49392b1f0eb424a64a4d8a10"
LCB_DATA_DIR = "/root/datasets/livecodebench_code_generation_lite/release_v6"
LCB_REPO_DIR = "LiveCodeBench"
LCB_REPO_REVISION = "28fef95ea8c9f7a547c8329f2cd3d32b92c1fa24"
LCB_NUM_PROCESSES = 4
LCB_TIMEOUT_SECONDS = 6
MAX_CONSECUTIVE_ERRORS = 3


In [ ]:
# Dependencies are imported normally in the next cell.
print("Starting Fed-MOPSD.", flush=True)


## Runtime, reproducibility, and official Qwen template

The teacher is fixed to the initial model revision throughout all clients and rounds, matching
OPSD's fixed-initial-policy implementation. The global and local students are full-parameter
models. No 4-bit or other model-weight quantization is used.


In [ ]:
import gc
import hashlib
import json
import math
import random
import re
import shutil
import subprocess
import sys
import time
from contextlib import ExitStack
from datetime import datetime, timezone
from pathlib import Path

# Import Unsloth before transformers so its supported kernels can be registered.
from unsloth import FastLanguageModel
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import disable_progress_bars, load_dataset
from safetensors import safe_open
from safetensors.torch import save_file
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from transformers.optimization import Adafactor

disable_progress_bars()
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for full-parameter training.")
STUDENT_DEVICE = torch.device("cuda:0")
TEACHER_DEVICE = torch.device("cuda:0")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.cuda.set_device(STUDENT_DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as exc:
    print(f"[WARN] Could not enable deterministic algorithms: {exc!r}", flush=True)

OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_CACHE_DIR = Path(CLIENT_DATA_DIR).expanduser().resolve()
TRAINING_DIR = OUTPUT_DIR / "training"
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(OUTPUT_DIR).free / 2**30
print(f"Output directory: {OUTPUT_DIR}", flush=True)
print(f"Free disk before training: {free_gb:.1f} GiB", flush=True)
if free_gb < MIN_FREE_DISK_GB_WARNING:
    print(
        f"[WARN] Full-parameter client checkpoints and the aggregate may need more than "
        f"{MIN_FREE_DISK_GB_WARNING} GiB; only {free_gb:.1f} GiB is currently free.",
        flush=True,
    )
print(f"Single visible device: logical cuda:0 -> physical GPU {os.environ['CUDA_VISIBLE_DEVICES']} "
      f"({torch.cuda.get_device_name(0)})", flush=True)
print(f"Student device: {STUDENT_DEVICE}; Teacher device: {TEACHER_DEVICE}", flush=True)
print(f"Training dtype: {DTYPE}; load_in_4bit=False; full-parameter=True", flush=True)

TRAIN_SOURCE_REVISIONS = {
    "math": "1f33e9dc2e8a1c639ca74f8024ad4a9f1f5eae62",
    "if": "fe0c7d350c9b4542b8d829a6f1daa1c259f0ba0e",
    "code": "3a7a0a0636ec96e3c1ec42ebe79ade467caa040d",
}
MODEL_SOURCE = str(Path(MODEL_SOURCE).expanduser().resolve())
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, use_fast=True, local_files_only=True)
tokenizer_config_path = Path(MODEL_SOURCE) / "tokenizer_config.json"
official_tokenizer_config = json.loads(Path(tokenizer_config_path).read_text(encoding="utf-8"))
tokenizer.chat_template = official_tokenizer_config.get("chat_template")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
audit_rendered = tokenizer.apply_chat_template(
    [{"role": "user", "content": "What is 1 + 1?"}], tokenize=False, add_generation_prompt=True
)
im_end_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
generation_eos_token_ids = [int(tokenizer.eos_token_id)]
if isinstance(im_end_token_id, int) and im_end_token_id >= 0 and im_end_token_id not in generation_eos_token_ids:
    generation_eos_token_ids.append(im_end_token_id)
template_audit = {
    "model": MODEL_NAME,
    "model_source": MODEL_SOURCE,
    "model_revision": MODEL_REVISION,
    "chat_template_source": str(tokenizer_config_path),
    "rendered_example": audit_rendered,
    "generation_eos_token_ids": generation_eos_token_ids,
    "uses_enable_thinking_argument": False,
}
(OUTPUT_DIR / "template_audit.json").write_text(
    json.dumps(template_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Loaded local tokenizer from {MODEL_SOURCE}", flush=True)


## Deterministic client datasets

The already prepared `math.jsonl`, `if.jsonl`, and `code.jsonl` files are used directly.
This notebook does not recount, redownload, or refilter those 3,000-row files.


In [ ]:
TRAIN_SOURCES = {
    "math": {
        "dataset": "siyanzhao/Openthoughts_math_30k_opsd",
        "config": None,
        "split": "train",
        "revision": TRAIN_SOURCE_REVISIONS["math"],
    },
    "if": {
        "dataset": "allenai/tulu-3-sft-personas-instruction-following",
        "config": None,
        "split": "train",
        "revision": TRAIN_SOURCE_REVISIONS["if"],
    },
    "code": {
        "dataset": "microsoft/rStar-Coder",
        "config": "seed_sft",
        "split": "train",
        "revision": TRAIN_SOURCE_REVISIONS["code"],
    },
}

def json_fingerprint(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:20]

DATA_SELECTION_FINGERPRINT = json_fingerprint({
    "sources": TRAIN_SOURCES,
    "client_samples": CLIENT_SAMPLES,
    "code_filter": "verified == True and is_passed == True",
    "math_filter": "correct is not False",
    "selection": "first-valid-records",
})

# The three normalized 3,000-row JSONL files were prepared once and are now used directly.
# No row counting, metadata validation, streaming download, or repeated filtering is performed here.
CLIENT_DATA_PATHS = ({} if LCB_EVAL_ONLY else {
    domain: DATA_CACHE_DIR / f"{domain}.jsonl" for domain in ("math", "if", "code")
})
print("Using local Client training files directly:", flush=True)
print(json.dumps({key: str(value) for key, value in CLIENT_DATA_PATHS.items()}, indent=2), flush=True)


## OPSD local objective

For every local record, the student samples one complete response from
`p_student(. | question)`. Both policies then score each next token under the same student
prefix. The frozen teacher additionally receives the reference answer as privileged context.
The loss is full-vocabulary forward KL, averaged across rollout tokens:

\[
\frac{1}{|\hat y|}\sum_t
D_{KL}\!\left(p_T(\cdot\mid x,y^*,\hat y_{<t})\,\|\,
p_S(\cdot\mid x,\hat y_{<t})\right).
\]

A custom autograd operator recomputes softmax chunks during backward. This preserves the exact
full-vocabulary gradient while avoiding persistent FP32 probability tensors of shape
`sequence_length × vocabulary_size`.


In [ ]:
CLIENT_LEARNING_RATES = {"math": LR_MATH, "if": LR_IF, "code": LR_CODE}

TRAINING_FINGERPRINT = json_fingerprint({
    "method": "fed-mopsd-full-parameter-v1",
    "model": MODEL_NAME,
    "model_source": MODEL_SOURCE,
    "model_revision": MODEL_REVISION,
    "data_fingerprint": DATA_SELECTION_FINGERPRINT,
    "client_samples": CLIENT_SAMPLES,
    "fed_rounds": FED_ROUNDS,
    "local_epochs": LOCAL_EPOCHS,
    "learning_rates": CLIENT_LEARNING_RATES,
    "weight_decay": WEIGHT_DECAY,
    "micro_batch_size": MICRO_BATCH_SIZE,
    "rollouts_per_prompt": ROLLOUTS_PER_PROMPT,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "checkpoint_every_optimizer_steps": CHECKPOINT_EVERY_OPTIMIZER_STEPS,
    "checkpoint_retention": KEEP_LAST_CHECKPOINTS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_train_seq_length": MAX_TRAIN_SEQ_LENGTH,
    "distill_temperature": DISTILL_TEMPERATURE,
    "seed": SEED,
    "teacher_policy": "fixed_initial_policy",
    "aggregation": DOMAIN_WEIGHTING,
})
print(f"Training fingerprint: {TRAINING_FINGERPRINT}", flush=True)
print(
    f"Batch config: prompts/micro-batch={MICRO_BATCH_SIZE}, rollouts/prompt={ROLLOUTS_PER_PROMPT}, "
    f"accumulation={GRADIENT_ACCUMULATION_STEPS}, "
    f"effective={EFFECTIVE_BATCH_SIZE} rollout sequences/optimizer-step",
    flush=True,
)

def format_seconds(seconds):
    seconds = int(max(0, seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def stable_seed(*parts):
    digest = hashlib.sha256("::".join(map(str, parts)).encode("utf-8")).digest()
    return (SEED + int.from_bytes(digest[:4], "big")) % (2**31 - 1)

def truncate_text(text, max_tokens):
    ids = tokenizer.encode(str(text), add_special_tokens=False)
    if len(ids) <= max_tokens:
        return str(text)
    return tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

def render_user_prompt(user_content):
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}], tokenize=False, add_generation_prompt=True
    )
    return tokenizer(rendered, return_tensors="pt", add_special_tokens=False)["input_ids"]

def build_student_prompt(question):
    question = truncate_text(question, MAX_QUESTION_TOKENS)
    prompt_ids = render_user_prompt(question)
    max_prompt_tokens = MAX_TRAIN_SEQ_LENGTH - MAX_NEW_TOKENS
    if prompt_ids.shape[1] > max_prompt_tokens:
        # Preserve the template while reducing only user content.
        allowed_question = max(32, MAX_QUESTION_TOKENS - (prompt_ids.shape[1] - max_prompt_tokens) - 16)
        question = truncate_text(question, allowed_question)
        prompt_ids = render_user_prompt(question)
    if prompt_ids.shape[1] > max_prompt_tokens:
        raise ValueError(f"Student prompt remains too long: {prompt_ids.shape[1]} > {max_prompt_tokens}")
    return question, prompt_ids

TEACHER_PREAMBLE = (
    "You are the privileged teacher in on-policy self-distillation. The student sees only the "
    "original request. You additionally receive a verified reference answer. Internalize the "
    "reference, then produce a fresh, correct answer to the original request. The assistant "
    "continuation may contain a student's imperfect prefix; at every position, predict the best "
    "next-token distribution for correcting or completing that answer.\n\n"
)

def build_teacher_prompt(question, reference_answer, rollout_length):
    reference_ids = tokenizer.encode(str(reference_answer), add_special_tokens=False)[:MAX_REFERENCE_TOKENS]
    max_prefix_tokens = MAX_TRAIN_SEQ_LENGTH - max(0, rollout_length - 1)

    def render_with_reference(token_count):
        reference = tokenizer.decode(reference_ids[:token_count], skip_special_tokens=True)
        content = (
            TEACHER_PREAMBLE
            + "Original request:\n" + question
            + "\n\nVerified reference answer (privileged):\n" + reference
            + "\n\nNow answer the original request."
        )
        return render_user_prompt(content)

    low, high = 0, len(reference_ids)
    best = None
    while low <= high:
        middle = (low + high) // 2
        candidate = render_with_reference(middle)
        if candidate.shape[1] <= max_prefix_tokens:
            best = candidate
            low = middle + 1
        else:
            high = middle - 1
    if best is None:
        raise ValueError("Teacher prompt cannot fit even after removing the reference answer.")
    return best

class ChunkedFullVocabForwardKL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, student_logits, teacher_logits, temperature, token_chunk):
        temperature = float(temperature)
        token_chunk = int(token_chunk)
        if student_logits.shape != teacher_logits.shape:
            raise ValueError((student_logits.shape, teacher_logits.shape))
        token_count = int(student_logits.shape[0])
        total = torch.zeros((), device=student_logits.device, dtype=torch.float32)
        with torch.no_grad():
            for start in range(0, token_count, token_chunk):
                stop = min(start + token_chunk, token_count)
                s_logp = F.log_softmax(student_logits[start:stop].float() / temperature, dim=-1)
                t_logp = F.log_softmax(teacher_logits[start:stop].float() / temperature, dim=-1)
                total += (t_logp.exp() * (t_logp - s_logp)).sum()
        ctx.save_for_backward(student_logits, teacher_logits)
        ctx.temperature = temperature
        ctx.token_chunk = token_chunk
        ctx.token_count = token_count
        return total * (temperature ** 2) / max(1, token_count)

    @staticmethod
    def backward(ctx, grad_output):
        student_logits, teacher_logits = ctx.saved_tensors
        gradient = torch.empty_like(student_logits)
        scale = ctx.temperature / max(1, ctx.token_count)
        with torch.no_grad():
            for start in range(0, ctx.token_count, ctx.token_chunk):
                stop = min(start + ctx.token_chunk, ctx.token_count)
                p_student = F.softmax(student_logits[start:stop].float() / ctx.temperature, dim=-1)
                p_teacher = F.softmax(teacher_logits[start:stop].float() / ctx.temperature, dim=-1)
                chunk_gradient = (p_student - p_teacher) * scale * grad_output.float()
                gradient[start:stop].copy_(chunk_gradient.to(student_logits.dtype))
        return gradient, None, None, None

def full_vocab_forward_kl(student_logits, teacher_logits):
    return ChunkedFullVocabForwardKL.apply(
        student_logits, teacher_logits, DISTILL_TEMPERATURE, KL_TOKEN_CHUNK
    )

def load_causal_lm(model_source, device, trainable):
    kwargs = {
        "pretrained_model_name_or_path": str(model_source),
        "dtype": DTYPE,
        "device_map": {"": int(device.index)},
        "low_cpu_mem_usage": True,
        "attn_implementation": "sdpa",
    }
    if Path(str(model_source)).expanduser().is_dir():
        kwargs["pretrained_model_name_or_path"] = str(Path(str(model_source)).expanduser().resolve())
        kwargs["local_files_only"] = True
    print(f"[MODEL LOAD] {kwargs['pretrained_model_name_or_path']} -> {device}", flush=True)
    model = AutoModelForCausalLM.from_pretrained(**kwargs)
    model.generation_config.max_length = None
    if trainable:
        model.requires_grad_(True)
        model.train()
        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model.requires_grad_(False)
        model.eval()
        model.config.use_cache = False
    return model

def forward_rollout_logits(model, prompt_ids, rollout_ids, device, requires_grad):
    rollout_length = int(rollout_ids.shape[1])
    if rollout_length < 1:
        raise ValueError("Empty rollout")
    input_ids = torch.cat([prompt_ids, rollout_ids[:, :-1]], dim=1).to(device)
    if input_ids.shape[1] > MAX_TRAIN_SEQ_LENGTH:
        raise ValueError(f"Training sequence too long: {input_ids.shape[1]}")
    attention_mask = torch.ones_like(input_ids)
    context = torch.enable_grad() if requires_grad else torch.no_grad()
    with context, torch.autocast(device_type="cuda", dtype=DTYPE):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            logits_to_keep=rollout_length,
            return_dict=True,
        )
    logits = outputs.logits
    if logits.shape[1] != rollout_length:
        logits = logits[:, -rollout_length:, :]
    assert logits.shape[1] == rollout_length
    return logits[0]

def generate_student_rollout(student, prompt_ids, sample_seed):
    random.seed(sample_seed)
    np.random.seed(sample_seed)
    torch.manual_seed(sample_seed)
    torch.cuda.manual_seed_all(sample_seed)
    student.eval()
    student.config.use_cache = True
    prompt_ids = prompt_ids.to(STUDENT_DEVICE)
    attention_mask = torch.ones_like(prompt_ids)
    with torch.inference_mode():
        generated = student.generate(
            input_ids=prompt_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TRAIN_TEMPERATURE,
            top_p=TRAIN_TOP_P,
            top_k=TRAIN_TOP_K,
            eos_token_id=generation_eos_token_ids,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    rollout = generated[:, prompt_ids.shape[1]:].detach().cpu()
    student.config.use_cache = False
    student.train()
    if rollout.shape[1] == 0:
        raise RuntimeError("Student generated an empty rollout.")
    return rollout

def iter_jsonl(path, start_index=0):
    with Path(path).open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            if index < start_index or not line.strip():
                continue
            yield index, json.loads(line)


## Client training and full-parameter FedAvg

Every 50 optimizer steps, each Client overwrites one latest-only checkpoint containing the model,
optimizer, scheduler, and next-sample position. A completed Client is skipped on rerun when its
fingerprint matches. Each local Student is retained under its
own `student_model/` directory and indexed by `client_students_manifest.json`. Aggregation reads sharded safetensors
tensor-by-tensor, accumulates floating tensors in FP32 on CPU, and writes BF16/FP16 tensors in
the original dtype; this avoids loading three complete 4B state dictionaries into RAM at once.


In [ ]:
def save_local_checkpoint(student, optimizer, scheduler, client_dir, state):
    # Latest-only policy: these fixed paths are overwritten; no checkpoint-N directories are created.
    model_dir = client_dir / "student_model"
    model_dir.mkdir(parents=True, exist_ok=True)
    old_use_cache = bool(student.config.use_cache)
    student.config.use_cache = True
    student.save_pretrained(
        model_dir, safe_serialization=True, max_shard_size=MODEL_SHARD_SIZE
    )
    tokenizer.save_pretrained(model_dir)
    student.config.use_cache = old_use_cache
    optimizer_path = client_dir / "optimizer.pt"
    scheduler_path = client_dir / "scheduler.pt"
    optimizer_tmp = client_dir / "optimizer.pt.tmp"
    scheduler_tmp = client_dir / "scheduler.pt.tmp"
    torch.save(optimizer.state_dict(), optimizer_tmp)
    torch.save(scheduler.state_dict(), scheduler_tmp)
    optimizer_tmp.replace(optimizer_path)
    scheduler_tmp.replace(scheduler_path)
    state = dict(state)
    state["checkpoint_policy"] = "latest_only"
    state["checkpoint_every_optimizer_steps"] = CHECKPOINT_EVERY_OPTIMIZER_STEPS
    state["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
    state_path = client_dir / "state.json"
    state_tmp = client_dir / "state.json.tmp"
    state_tmp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    state_tmp.replace(state_path)
    print(
        f"[CHECKPOINT latest-only] {state['domain']} round={state['round']} "
        f"next_sample={state['next_sample_index']} optimizer_step={state['optimizer_step']}",
        flush=True,
    )

def train_one_client(domain, round_index, global_source, teacher):
    client_dir = TRAINING_DIR / f"round_{round_index:03d}" / f"client_{domain}"
    client_dir.mkdir(parents=True, exist_ok=True)
    state_path = client_dir / "state.json"
    previous_state = {}
    student_model_dir = client_dir / "student_model"
    if RESUME and state_path.is_file():
        previous_state = json.loads(state_path.read_text(encoding="utf-8"))
        if previous_state.get("training_fingerprint") != TRAINING_FINGERPRINT:
            raise RuntimeError(f"Fingerprint mismatch in {state_path}; use a new OUTPUT_DIR.")
        if previous_state.get("status") == "complete":
            if not (student_model_dir / "config.json").is_file():
                raise FileNotFoundError(f"Complete state exists but local Student is missing: {student_model_dir}")
            print(f"[SKIP] train/{domain}/round-{round_index}: complete local Student exists", flush=True)
            return student_model_dir

    if previous_state:
        required_resume_files = [
            student_model_dir / "config.json",
            client_dir / "optimizer.pt",
            client_dir / "scheduler.pt",
        ]
        missing_resume_files = [str(path) for path in required_resume_files if not path.is_file()]
        if missing_resume_files:
            raise RuntimeError(
                "Latest checkpoint is incomplete and cannot be resumed; missing: "
                + ", ".join(missing_resume_files)
            )
    resume_local = bool(previous_state)
    student_source = student_model_dir if resume_local else global_source
    print(f"[LOAD] trainable {domain} student from {student_source}", flush=True)
    student = load_causal_lm(student_source, STUDENT_DEVICE, trainable=True)
    total_parameters = sum(parameter.numel() for parameter in student.parameters())
    trainable_parameters = sum(parameter.numel() for parameter in student.parameters() if parameter.requires_grad)
    assert trainable_parameters == total_parameters, (
        f"Full-parameter contract violated: trainable={trainable_parameters}, total={total_parameters}"
    )
    print(f"[FULL-FT] {domain}: {trainable_parameters:,}/{total_parameters:,} parameters trainable", flush=True)

    learning_rate = CLIENT_LEARNING_RATES[domain]
    optimizer = Adafactor(
        student.parameters(),
        lr=learning_rate,
        scale_parameter=False,
        relative_step=False,
        warmup_init=False,
        weight_decay=WEIGHT_DECAY,
    )
    total_optimizer_steps = math.ceil(CLIENT_SAMPLES / GRADIENT_ACCUMULATION_STEPS)
    warmup_steps = max(1, int(total_optimizer_steps * WARMUP_RATIO))
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_optimizer_steps
    )
    start_index = int(previous_state.get("next_sample_index", 0)) if resume_local else 0
    optimizer_step = int(previous_state.get("optimizer_step", 0)) if resume_local else 0
    if resume_local:
        optimizer.load_state_dict(torch.load(client_dir / "optimizer.pt", map_location=STUDENT_DEVICE, weights_only=False))
        scheduler.load_state_dict(torch.load(client_dir / "scheduler.pt", map_location="cpu", weights_only=False))
        print(
            f"[RESUME latest-only] train/{domain}/round-{round_index}: sample={start_index}, "
            f"optimizer_step={optimizer_step}",
            flush=True,
        )

    optimizer.zero_grad(set_to_none=True)
    training_log_path = client_dir / "training_log.jsonl"
    optimizer_step_log_path = client_dir / "optimizer_step_losses.jsonl"
    if not resume_local:
        for stale_log_path in (training_log_path, optimizer_step_log_path):
            if stale_log_path.is_file():
                stale_log_path.unlink()
    if resume_local and training_log_path.is_file():
        # A crash can leave log rows newer than the last committed checkpoint. Remove those rows
        # so resumed samples are represented exactly once in the final training log.
        retained_log_lines = []
        with training_log_path.open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                logged = json.loads(line)
                same_run = (
                    logged.get("training_fingerprint") == TRAINING_FINGERPRINT
                    and logged.get("round") == round_index
                    and logged.get("domain") == domain
                )
                if same_run and int(logged.get("sample_index", -1)) >= start_index:
                    continue
                retained_log_lines.append(line if line.endswith("\n") else line + "\n")
        log_tmp = training_log_path.with_suffix(".jsonl.tmp")
        with log_tmp.open("w", encoding="utf-8") as handle:
            handle.writelines(retained_log_lines)
        log_tmp.replace(training_log_path)
        print(f"[RESUME] pruned training log to committed sample index {start_index}", flush=True)
    if resume_local and optimizer_step_log_path.is_file():
        retained_step_lines = []
        with optimizer_step_log_path.open("r", encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                logged = json.loads(line)
                if int(logged.get("optimizer_step", -1)) > optimizer_step:
                    continue
                retained_step_lines.append(line if line.endswith("\n") else line + "\n")
        step_log_tmp = optimizer_step_log_path.with_suffix(".jsonl.tmp")
        with step_log_tmp.open("w", encoding="utf-8") as handle:
            handle.writelines(retained_step_lines)
        step_log_tmp.replace(optimizer_step_log_path)
        print(f"[RESUME] pruned step-loss log to optimizer step {optimizer_step}", flush=True)
    if resume_local:
        committed_step_records = []
        if optimizer_step_log_path.is_file():
            with optimizer_step_log_path.open("r", encoding="utf-8") as handle:
                committed_step_records = [json.loads(line) for line in handle if line.strip()]
        committed_steps = [int(record.get("optimizer_step", -1)) for record in committed_step_records]
        if committed_steps != list(range(1, optimizer_step + 1)):
            # Migration path for checkpoints created before optimizer-step loss logging was added.
            if not training_log_path.is_file():
                raise RuntimeError("Cannot reconstruct committed step losses without training_log.jsonl")
            samples_by_index = {}
            with training_log_path.open("r", encoding="utf-8") as handle:
                for line in handle:
                    if not line.strip():
                        continue
                    sample_record = json.loads(line)
                    same_run = (
                        sample_record.get("training_fingerprint") == TRAINING_FINGERPRINT
                        and sample_record.get("round") == round_index
                        and sample_record.get("domain") == domain
                    )
                    sample_position = int(sample_record.get("sample_index", -1))
                    if same_run and 0 <= sample_position < start_index:
                        samples_by_index[sample_position] = sample_record
            if sorted(samples_by_index) != list(range(start_index)):
                raise RuntimeError(
                    f"Cannot reconstruct {domain} step losses: committed sample log is incomplete."
                )
            rebuilt_records = []
            for committed_step in range(1, optimizer_step + 1):
                first_index = (committed_step - 1) * GRADIENT_ACCUMULATION_STEPS
                stop_index = min(committed_step * GRADIENT_ACCUMULATION_STEPS, start_index)
                step_samples = [samples_by_index[index] for index in range(first_index, stop_index)]
                if not step_samples:
                    raise RuntimeError(f"No samples available for reconstructed step {committed_step}")
                last_sample = step_samples[-1]
                rebuilt_records.append({
                    "training_fingerprint": TRAINING_FINGERPRINT,
                    "round": round_index,
                    "domain": domain,
                    "optimizer_step": committed_step,
                    "loss": sum(float(item["loss"]) for item in step_samples) / len(step_samples),
                    "samples_in_step": len(step_samples),
                    "last_sample_index": int(last_sample["sample_index"]),
                    "learning_rate": float(last_sample["learning_rate"]),
                    "grad_norm": last_sample.get("grad_norm"),
                    "elapsed_seconds": None,
                    "reconstructed_from_sample_log": True,
                })
            step_log_tmp = optimizer_step_log_path.with_suffix(".jsonl.tmp")
            with step_log_tmp.open("w", encoding="utf-8") as handle:
                for rebuilt_record in rebuilt_records:
                    handle.write(json.dumps(rebuilt_record, ensure_ascii=False) + "\n")
            step_log_tmp.replace(optimizer_step_log_path)
            print(f"[RESUME] reconstructed {optimizer_step} committed step losses", flush=True)
    started = time.time()
    accumulation_count = 0
    step_loss_sum = 0.0
    step_sample_count = 0
    for sample_index, record in iter_jsonl(CLIENT_DATA_PATHS[domain], start_index):
        if sample_index >= CLIENT_SAMPLES:
            break
        question, student_prompt_ids = build_student_prompt(record["question"])
        rollout_losses = []
        rollout_token_counts = []
        rollout_seeds = []
        for rollout_index in range(ROLLOUTS_PER_PROMPT):
            rollout_seed = stable_seed(
                TRAINING_FINGERPRINT, round_index, domain, sample_index, rollout_index
            )
            rollout_ids = generate_student_rollout(student, student_prompt_ids, rollout_seed)
            teacher_prompt_ids = build_teacher_prompt(
                question, record["reference_answer"], int(rollout_ids.shape[1])
            )

            teacher_logits = forward_rollout_logits(
                teacher, teacher_prompt_ids, rollout_ids, TEACHER_DEVICE, requires_grad=False
            ).to(STUDENT_DEVICE, non_blocking=True)
            student_logits = forward_rollout_logits(
                student, student_prompt_ids, rollout_ids, STUDENT_DEVICE, requires_grad=True
            )
            raw_loss = full_vocab_forward_kl(student_logits, teacher_logits)
            (raw_loss / (GRADIENT_ACCUMULATION_STEPS * ROLLOUTS_PER_PROMPT)).backward()
            rollout_losses.append(float(raw_loss.detach().cpu()))
            rollout_token_counts.append(int(rollout_ids.shape[1]))
            rollout_seeds.append(rollout_seed)
            del teacher_logits, student_logits, raw_loss, rollout_ids, teacher_prompt_ids

        loss_value = sum(rollout_losses) / len(rollout_losses)
        mean_rollout_tokens = sum(rollout_token_counts) / len(rollout_token_counts)
        accumulation_count += 1
        step_loss_sum += loss_value
        step_sample_count += 1

        is_last = sample_index + 1 >= CLIENT_SAMPLES
        if accumulation_count >= GRADIENT_ACCUMULATION_STEPS or is_last:
            step_loss_value = step_loss_sum / max(1, step_sample_count)
            samples_in_step = step_sample_count
            grad_norm = float(torch.nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM))
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            optimizer_step += 1
            step_record = {
                "training_fingerprint": TRAINING_FINGERPRINT,
                "round": round_index,
                "domain": domain,
                "optimizer_step": optimizer_step,
                "loss": step_loss_value,
                "samples_in_step": samples_in_step,
                "last_sample_index": sample_index,
                "learning_rate": float(scheduler.get_last_lr()[0]),
                "grad_norm": grad_norm,
                "elapsed_seconds": time.time() - started,
            }
            with optimizer_step_log_path.open("a", encoding="utf-8") as handle:
                handle.write(json.dumps(step_record, ensure_ascii=False) + "\n")
                handle.flush()
            accumulation_count = 0
            step_loss_sum = 0.0
            step_sample_count = 0
        else:
            grad_norm = None
            step_loss_value = None

        log_record = {
            "training_fingerprint": TRAINING_FINGERPRINT,
            "round": round_index,
            "domain": domain,
            "sample_index": sample_index,
            "sample_id": record["sample_id"],
            "loss": loss_value,
            "rollouts_per_prompt": ROLLOUTS_PER_PROMPT,
            "rollout_losses": rollout_losses,
            "rollout_token_counts": rollout_token_counts,
            "mean_rollout_tokens": mean_rollout_tokens,
            "optimizer_step": optimizer_step,
            "learning_rate": float(scheduler.get_last_lr()[0]),
            "grad_norm": grad_norm,
            "rollout_seeds": rollout_seeds,
        }
        with training_log_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(log_record, ensure_ascii=False) + "\n")
            handle.flush()
        done = sample_index + 1
        if step_loss_value is not None:
            print(
                f"[STEP LOSS] train/{domain}/round-{round_index}: "
                f"step={optimizer_step}/{total_optimizer_steps} loss={step_loss_value:.6f} "
                f"samples={samples_in_step}",
                flush=True,
            )
        print(
            f"[PROGRESS] train/{domain}/round-{round_index}: {done}/{CLIENT_SAMPLES} "
            f"({100 * done / CLIENT_SAMPLES:.2f}%) loss={loss_value:.6f} "
            f"rollouts={ROLLOUTS_PER_PROMPT} mean_tokens={mean_rollout_tokens:.1f} "
            f"opt_step={optimizer_step}/{total_optimizer_steps} "
            f"elapsed={format_seconds(time.time() - started)}",
            flush=True,
        )

        should_checkpoint = (
            accumulation_count == 0
            and CHECKPOINT_EVERY_OPTIMIZER_STEPS > 0
            and optimizer_step > 0
            and optimizer_step % CHECKPOINT_EVERY_OPTIMIZER_STEPS == 0
            and not is_last
        )
        if should_checkpoint:
            save_local_checkpoint(student, optimizer, scheduler, client_dir, {
                "status": "in_progress",
                "training_fingerprint": TRAINING_FINGERPRINT,
                "domain": domain,
                "round": round_index,
                "next_sample_index": done,
                "optimizer_step": optimizer_step,
                "global_source": str(global_source),
                "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            })

    if start_index >= CLIENT_SAMPLES:
        raise RuntimeError(f"Incomplete state for {domain} has next_sample_index={start_index}.")
    final_state = {
        "status": "complete",
        "training_fingerprint": TRAINING_FINGERPRINT,
        "domain": domain,
        "round": round_index,
        "next_sample_index": CLIENT_SAMPLES,
        "optimizer_step": optimizer_step,
        "learning_rate": learning_rate,
        "global_source": str(global_source),
        "student_checkpoint": str(student_model_dir.resolve()),
        "full_parameter_training": True,
        "teacher_model": MODEL_NAME,
        "teacher_revision": MODEL_REVISION,
        "teacher_policy": "fixed_initial_policy_with_privileged_reference",
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    save_local_checkpoint(student, optimizer, scheduler, client_dir, final_state)
    del optimizer, scheduler, student
    gc.collect()
    torch.cuda.empty_cache()
    return student_model_dir

def checkpoint_weight_map(model_dir):
    model_dir = Path(model_dir)
    index_path = model_dir / "model.safetensors.index.json"
    if index_path.is_file():
        payload = json.loads(index_path.read_text(encoding="utf-8"))
        return dict(payload["weight_map"])
    single = model_dir / "model.safetensors"
    if not single.is_file():
        candidates = sorted(model_dir.glob("*.safetensors"))
        if len(candidates) != 1:
            raise FileNotFoundError(f"No unambiguous safetensors checkpoint in {model_dir}")
        single = candidates[0]
    with safe_open(single, framework="pt", device="cpu") as reader:
        return {key: single.name for key in reader.keys()}

def federated_average_checkpoints(client_model_dirs, output_dir, weights):
    output_dir = Path(output_dir)
    state_path = output_dir / "fedavg_state.json"
    if RESUME and state_path.is_file():
        state = json.loads(state_path.read_text(encoding="utf-8"))
        if state.get("status") == "complete" and state.get("training_fingerprint") == TRAINING_FINGERPRINT:
            print(f"[SKIP] FedAvg aggregate already exists: {output_dir}", flush=True)
            return output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    if not math.isclose(sum(weights), 1.0, rel_tol=0.0, abs_tol=1e-12):
        raise ValueError(weights)
    maps = [checkpoint_weight_map(path) for path in client_model_dirs]
    key_set = set(maps[0])
    if any(set(mapping) != key_set for mapping in maps[1:]):
        raise RuntimeError("Client checkpoints do not have identical parameter keys.")

    with ExitStack() as stack:
        readers = []
        for model_dir, mapping in zip(client_model_dirs, maps):
            shard_readers = {
                shard: stack.enter_context(
                    safe_open(Path(model_dir) / shard, framework="pt", device="cpu")
                )
                for shard in sorted(set(mapping.values()))
            }
            readers.append(shard_readers)

        output_weight_map = {}
        current_shard = {}
        current_bytes = 0
        shard_number = 0
        total_size = 0

        def flush_shard():
            nonlocal current_shard, current_bytes, shard_number
            if not current_shard:
                return
            shard_number += 1
            filename = f"model-{shard_number:05d}.safetensors"
            save_file(current_shard, output_dir / filename, metadata={"format": "pt"})
            for tensor_name in current_shard:
                output_weight_map[tensor_name] = filename
            print(
                f"[FEDAVG] wrote {filename}: {len(current_shard)} tensors, "
                f"{current_bytes / 2**30:.2f} GiB",
                flush=True,
            )
            current_shard = {}
            current_bytes = 0

        keys = sorted(key_set)
        started = time.time()
        for key_index, key in enumerate(keys, 1):
            tensors = [
                readers[index][maps[index][key]].get_tensor(key)
                for index in range(len(client_model_dirs))
            ]
            if tensors[0].is_floating_point():
                averaged = torch.zeros_like(tensors[0], dtype=torch.float32)
                for weight, tensor in zip(weights, tensors):
                    averaged.add_(tensor.float(), alpha=float(weight))
                averaged = averaged.to(tensors[0].dtype)
            else:
                averaged = tensors[0].clone()
            tensor_bytes = averaged.numel() * averaged.element_size()
            if current_shard and current_bytes + tensor_bytes > FEDAVG_SHARD_SIZE_BYTES:
                flush_shard()
            current_shard[key] = averaged.contiguous()
            current_bytes += tensor_bytes
            total_size += tensor_bytes
            del tensors, averaged
            print(
                f"[PROGRESS] FedAvg: {key_index}/{len(keys)} ({100 * key_index / len(keys):.2f}%) "
                f"elapsed={format_seconds(time.time() - started)}",
                flush=True,
            )
        flush_shard()

    index_payload = {
        "metadata": {"total_size": total_size},
        "weight_map": output_weight_map,
    }
    (output_dir / "model.safetensors.index.json").write_text(
        json.dumps(index_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    for filename in ("config.json", "generation_config.json"):
        source_file = Path(client_model_dirs[0]) / filename
        if source_file.is_file():
            shutil.copy2(source_file, output_dir / filename)
    tokenizer.save_pretrained(output_dir)
    fedavg_state = {
        "status": "complete",
        "training_fingerprint": TRAINING_FINGERPRINT,
        "clients": [str(path) for path in client_model_dirs],
        "weights": weights,
        "parameter_tensors": len(keys),
        "total_size_bytes": total_size,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    state_path.write_text(json.dumps(fedavg_state, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[FEDAVG] complete: {output_dir}", flush=True)
    return output_dir


## Single-GPU Client execution setup

Physical GPU 1 is the only visible CUDA device and therefore appears inside Python as logical
`cuda:0`. The frozen Teacher is loaded once and reused. Each Client cell below independently
starts from the same `ROUND_GLOBAL_SOURCE`; no Client starts from another Client's Student.


In [ ]:
ROUND_INDEX = 0
ROUND_GLOBAL_SOURCE = MODEL_SOURCE
ROUND_OUTPUT = TRAINING_DIR / "global_round_001"
FINAL_MODEL_DIR = ROUND_OUTPUT
final_state_path = FINAL_MODEL_DIR / "fedavg_state.json"
final_complete = False
if final_state_path.is_file():
    final_state = json.loads(final_state_path.read_text(encoding="utf-8"))
    final_complete = (
        final_state.get("status") == "complete"
        and final_state.get("training_fingerprint") == TRAINING_FINGERPRINT
    )
if LCB_EVAL_ONLY and not final_complete:
    raise FileNotFoundError(
        f"LCB_EVAL_ONLY requires an existing completed checkpoint at {FINAL_MODEL_DIR}."
    )

def expected_local_student(domain):
    return TRAINING_DIR / f"round_{ROUND_INDEX:03d}" / f"client_{domain}" / "student_model"

def local_student_is_complete(domain):
    client_dir = TRAINING_DIR / f"round_{ROUND_INDEX:03d}" / f"client_{domain}"
    state_path = client_dir / "state.json"
    model_dir = client_dir / "student_model"
    if not state_path.is_file() or not (model_dir / "config.json").is_file():
        return False
    state = json.loads(state_path.read_text(encoding="utf-8"))
    return (state.get("status") == "complete"
            and state.get("training_fingerprint") == TRAINING_FINGERPRINT)

PENDING_CLIENTS = [
    domain for domain in ("math", "if", "code")
    if not local_student_is_complete(domain)
]
teacher = None
if PENDING_CLIENTS and not final_complete and not LCB_EVAL_ONLY:
    print(f"[LOAD] frozen initial Teacher on {TEACHER_DEVICE}; pending={PENDING_CLIENTS}", flush=True)
    teacher = load_causal_lm(MODEL_SOURCE, TEACHER_DEVICE, trainable=False)
else:
    print(f"[SKIP] No Teacher load required; pending={PENDING_CLIENTS}, final_complete={final_complete}", flush=True)

def run_explicit_client_cell(domain):
    if final_complete:
        model_dir = expected_local_student(domain)
        if not (model_dir / "config.json").is_file():
            raise FileNotFoundError(f"FedAvg is complete but saved local Student is missing: {model_dir}")
        print(f"[SKIP] {domain} Client: final aggregate and local Student already exist", flush=True)
        return model_dir.resolve()
    print(f"[CLIENT START] {domain}: initializing from {ROUND_GLOBAL_SOURCE}", flush=True)
    model_dir = train_one_client(domain, ROUND_INDEX, ROUND_GLOBAL_SOURCE, teacher)
    if not (Path(model_dir) / "config.json").is_file():
        raise FileNotFoundError(f"{domain} Client did not produce a complete Student: {model_dir}")
    print(f"[CLIENT COMPLETE] {domain}: {model_dir}", flush=True)
    return Path(model_dir).resolve()


## Client 1 — Math OPSD

This cell trains only the Math Client and saves its complete Student before returning.


In [ ]:
MATH_STUDENT_DIR = run_explicit_client_cell("math")
print(f"Math Student checkpoint: {MATH_STUDENT_DIR}", flush=True)


## Client 2 — Instruction-following OPSD

This cell independently initializes from the same global checkpoint, trains only the IF Client,
and saves its complete Student.


In [ ]:
IF_STUDENT_DIR = run_explicit_client_cell("if")
print(f"IF Student checkpoint: {IF_STUDENT_DIR}", flush=True)


## Client 3 — Code OPSD

This cell independently initializes from the same global checkpoint, trains only the Code Client,
and saves its complete Student.


In [ ]:
CODE_STUDENT_DIR = run_explicit_client_cell("code")
print(f"Code Student checkpoint: {CODE_STUDENT_DIR}", flush=True)


## Client optimizer-step loss records and curve

Each point is the mean forward-KL loss of the microbatches contributing to one
`optimizer.step()`. Raw per-Client JSONL files are retained beside each local Student; this cell
also exports a combined CSV/JSON file and one three-domain loss chart.


In [ ]:
LOSS_RESULTS_DIR = OUTPUT_DIR / "training_losses"
LOSS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
step_loss_frames = []
for domain in ("math", "if", "code"):
    loss_log_path = (
        TRAINING_DIR / f"round_{ROUND_INDEX:03d}" / f"client_{domain}"
        / "optimizer_step_losses.jsonl"
    )
    if not loss_log_path.is_file():
        raise FileNotFoundError(f"Missing optimizer-step loss log for {domain}: {loss_log_path}")
    records = []
    with loss_log_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                records.append(json.loads(line))
    records = [
        record for record in records
        if record.get("training_fingerprint") == TRAINING_FINGERPRINT
        and int(record.get("round", -1)) == ROUND_INDEX
        and record.get("domain") == domain
    ]
    records.sort(key=lambda record: int(record["optimizer_step"]))
    frame = pd.DataFrame(records)
    frame["smoothed_loss"] = frame["loss"].rolling(
        window=LOSS_CURVE_SMOOTHING_WINDOW, min_periods=1
    ).mean()
    frame["smoothed_grad_norm"] = frame["grad_norm"].rolling(
        window=LOSS_CURVE_SMOOTHING_WINDOW, min_periods=1
    ).mean()
    step_loss_frames.append(frame)

combined_step_losses = pd.concat(step_loss_frames, ignore_index=True)
LOSS_CSV_PATH = LOSS_RESULTS_DIR / "client_optimizer_step_losses.csv"
LOSS_JSON_PATH = LOSS_RESULTS_DIR / "client_optimizer_step_losses.json"
LOSS_FIGURE_PATH = LOSS_RESULTS_DIR / "client_optimizer_step_loss_curve.png"
GRADIENT_FIGURE_PATH = LOSS_RESULTS_DIR / "client_optimizer_step_gradient_norm_curve.png"
combined_step_losses.to_csv(LOSS_CSV_PATH, index=False, encoding="utf-8-sig")
combined_step_losses.to_json(
    LOSS_JSON_PATH, orient="records", force_ascii=False, indent=2
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

domain_labels = {"math": "Math", "if": "Instruction Following", "code": "Code"}
domain_colors = {"math": "#4C78A8", "if": "#F58518", "code": "#54A24B"}
fig, ax = plt.subplots(figsize=(11, 6))
for domain in ("math", "if", "code"):
    frame = combined_step_losses[combined_step_losses["domain"] == domain].sort_values("optimizer_step")
    color = domain_colors[domain]
    ax.plot(frame["optimizer_step"], frame["loss"], color=color, alpha=0.20, linewidth=0.8)
    ax.plot(
        frame["optimizer_step"], frame["smoothed_loss"], color=color, linewidth=2.0,
        label=f"{domain_labels[domain]} (moving mean {LOSS_CURVE_SMOOTHING_WINDOW})",
    )
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Mean forward-KL loss")
ax.set_title("FedAvg-MOPSD local Client training losses")
ax.grid(True, linestyle="--", alpha=0.30)
ax.legend()
fig.tight_layout()
fig.savefig(LOSS_FIGURE_PATH, dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(11, 6))
for domain in ("math", "if", "code"):
    frame = combined_step_losses[combined_step_losses["domain"] == domain].sort_values("optimizer_step")
    color = domain_colors[domain]
    ax.plot(frame["optimizer_step"], frame["grad_norm"], color=color, alpha=0.20, linewidth=0.8)
    ax.plot(
        frame["optimizer_step"], frame["smoothed_grad_norm"], color=color, linewidth=2.0,
        label=f"{domain_labels[domain]} (moving mean {LOSS_CURVE_SMOOTHING_WINDOW})",
    )
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Gradient norm before clipping")
ax.set_title("FedAvg-MOPSD local Client gradient norms")
ax.grid(True, linestyle="--", alpha=0.30)
ax.legend()
fig.tight_layout()
fig.savefig(GRADIENT_FIGURE_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Step-loss CSV:  {LOSS_CSV_PATH}", flush=True)
print(f"Step-loss JSON: {LOSS_JSON_PATH}", flush=True)
print(f"Loss curve:     {LOSS_FIGURE_PATH}", flush=True)
print(f"Gradient curve: {GRADIENT_FIGURE_PATH}", flush=True)


## Server — FedAvg aggregation

This cell runs only after all three local Students exist. It averages their full parameters with
equal weights and writes the global Student plus a manifest of all retained local checkpoints.


In [ ]:
client_models = [MATH_STUDENT_DIR, IF_STUDENT_DIR, CODE_STUDENT_DIR]
for client_model in client_models:
    if not (Path(client_model) / "config.json").is_file():
        raise FileNotFoundError(f"Cannot aggregate missing local Student: {client_model}")
if DOMAIN_WEIGHTING != "equal":
    raise ValueError("This notebook implements the declared equal-domain baseline only.")
weights = [1.0 / len(client_models)] * len(client_models)
if final_complete:
    print(f"[SKIP] FedAvg aggregate already complete: {FINAL_MODEL_DIR}", flush=True)
else:
    FINAL_MODEL_DIR = federated_average_checkpoints(client_models, ROUND_OUTPUT, weights).resolve()

if teacher is not None:
    del teacher
    teacher = None
gc.collect()
torch.cuda.empty_cache()
assert (FINAL_MODEL_DIR / "fedavg_state.json").is_file(), FINAL_MODEL_DIR

client_student_entries = []
for domain, student_model_dir in zip(("math", "if", "code"), client_models):
    client_dir = TRAINING_DIR / f"round_{ROUND_INDEX:03d}" / f"client_{domain}"
    state_path = client_dir / "state.json"
    if not state_path.is_file():
        raise FileNotFoundError(f"Missing completed Client state: {state_path}")
    client_state = json.loads(state_path.read_text(encoding="utf-8"))
    if client_state.get("status") != "complete":
        raise RuntimeError(f"Client is not complete: {state_path}")
    if client_state.get("training_fingerprint") != TRAINING_FINGERPRINT:
        raise RuntimeError(f"Client fingerprint mismatch: {state_path}")
    source = TRAIN_SOURCES[domain]
    client_student_entries.append({
        "round": ROUND_INDEX,
        "domain": domain,
        "student_checkpoint": str(Path(student_model_dir).resolve()),
        "state_path": str(state_path.resolve()),
        "dataset": source["dataset"],
        "dataset_config": source["config"],
        "dataset_split": source["split"],
        "dataset_revision": source["revision"],
        "samples": CLIENT_SAMPLES,
        "learning_rate": CLIENT_LEARNING_RATES[domain],
        "global_source": client_state.get("global_source"),
        "optimizer_steps": client_state.get("optimizer_step"),
        "status": client_state["status"],
        "optimizer_step_loss_log": str((client_dir / "optimizer_step_losses.jsonl").resolve()),
        "full_parameter_training": True,
    })

CLIENT_STUDENTS_MANIFEST = TRAINING_DIR / "client_students_manifest.json"
client_students_manifest = {
    "status": "complete",
    "training_fingerprint": TRAINING_FINGERPRINT,
    "base_model": MODEL_NAME,
    "base_model_revision": MODEL_REVISION,
    "teacher": {
        "model": MODEL_NAME,
        "revision": MODEL_REVISION,
        "policy": "fixed_initial_policy_with_privileged_reference",
    },
    "aggregation": "FedAvg",
    "aggregation_weights": dict(zip(("math", "if", "code"), weights)),
    "final_aggregated_student": str(FINAL_MODEL_DIR.resolve()),
    "loss_artifacts": {
        "combined_csv": str(LOSS_CSV_PATH.resolve()),
        "combined_json": str(LOSS_JSON_PATH.resolve()),
        "curve_png": str(LOSS_FIGURE_PATH.resolve()),
        "smoothing_window": LOSS_CURVE_SMOOTHING_WINDOW,
    },
    "local_students": client_student_entries,
    "written_at_utc": datetime.now(timezone.utc).isoformat(),
}
CLIENT_STUDENTS_MANIFEST.write_text(
    json.dumps(client_students_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Saved local Student manifest: {CLIENT_STUDENTS_MANIFEST}", flush=True)
print(f"Final aggregated Student: {FINAL_MODEL_DIR}", flush=True)


## Load the aggregated student for evaluation

Only the global FedAvg checkpoint is evaluated here. Every local Student remains saved under
`training/round_NNN/client_{math,if,code}/student_model`, and
`training/client_students_manifest.json` records the exact paths and training metadata for later
domain-specialization evaluation.


In [ ]:
EVAL_OUTPUT_DIR = OUTPUT_DIR / "evaluation"
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR = EVAL_OUTPUT_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
EVAL_MODEL_NAME = str(FINAL_MODEL_DIR)

if LCB_EVAL_ONLY:
    RUN_MATH = False
    RUN_IF = False
    RUN_LCB = True
    RUN_LCB_EVALUATION = True
    print("[MODE] LCB_EVAL_ONLY: skip LLM loading and reuse saved code generations.", flush=True)
else:
    print(f"Loading aggregated student with Unsloth from {EVAL_MODEL_NAME}", flush=True)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=EVAL_MODEL_NAME,
        max_seq_length=EVAL_MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=False,
        device_map={"": 0},
    )
    FastLanguageModel.for_inference(model)
    model.eval()
    model.generation_config.max_length = None
    # Unsloth may replace Qwen's valid <|endoftext|> padding token while loading.
    # Restore the tokenizer/model padding IDs explicitly before generation.
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    model.config.pad_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.eos_token_id
    assert not bool(getattr(model, "is_loaded_in_4bit", False))
    assert getattr(model.config, "quantization_config", None) is None
    parameter_devices = {str(parameter.device) for parameter in model.parameters()}
    assert parameter_devices == {"cuda:0"}, f"Evaluation model escaped cuda:0: {parameter_devices}"
    if not tokenizer.chat_template:
        tokenizer.chat_template = official_tokenizer_config["chat_template"]
    eval_audit = tokenizer.apply_chat_template(
        [{"role": "user", "content": "What is 1 + 1?"}], tokenize=False, add_generation_prompt=True
    )
    print(f"Evaluation CUDA allocation: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB", flush=True)


## Shared generation and persistence helpers

Each completed sample is appended immediately to JSONL and flushed to disk. A run fingerprint prevents resume mode from silently mixing different models, prompts, or decoding settings. No `tqdm` progress bar is used.


In [ ]:
PROMPT_VERSION = "fed-mopsd-global-eval-v1"
fingerprint_payload = {
    "model": EVAL_MODEL_NAME,
    "training_fingerprint": TRAINING_FINGERPRINT,
    "prompt_version": PROMPT_VERSION,
    "max_seq_length": EVAL_MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "min_p": MIN_P,
    "seed": SEED,
    "model_revision": MODEL_REVISION,
}
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"Run fingerprint: {RUN_FINGERPRINT}", flush=True)

def format_seconds(seconds):
    seconds = int(max(0, seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def print_progress(label, done, total, started_at):
    percent = 100.0 * done / total if total else 100.0
    print(
        f"[PROGRESS] {label}: {done}/{total} ({percent:.2f}%) "
        f"elapsed={format_seconds(time.time() - started_at)}",
        flush=True,
    )

def stable_seed(sample_id):
    digest = hashlib.sha256(str(sample_id).encode("utf-8")).digest()
    return (SEED + int.from_bytes(digest[:4], "big")) % (2**31 - 1)

def split_thinking(generated_text):
    text = generated_text.strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False

def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

def load_matching_records(path):
    latest = {}
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"[WARN] Ignoring truncated JSONL line {path}:{line_number}", flush=True)
                continue
            if record.get("run_fingerprint") == RUN_FINGERPRINT:
                latest[str(record["sample_id"])] = record
    return latest

def limit_dataset(dataset, limit):
    if limit is None:
        return dataset
    return dataset.select(range(min(int(limit), len(dataset))))

def generate_one(user_prompt, max_new_tokens, sample_id, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    encoded = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to("cuda:0")
    attention_mask = encoded["attention_mask"].to("cuda:0")
    input_tokens = int(input_ids.shape[1])
    available = EVAL_MAX_SEQ_LENGTH - input_tokens
    if available < 1:
        raise ValueError(f"Prompt has {input_tokens} tokens and exceeds EVAL_MAX_SEQ_LENGTH={EVAL_MAX_SEQ_LENGTH}.")
    effective_max_new_tokens = min(int(max_new_tokens), int(available))
    if effective_max_new_tokens < max_new_tokens:
        print(
            f"[WARN] {sample_id}: capped max_new_tokens from {max_new_tokens} "
            f"to {effective_max_new_tokens} to fit the context window.",
            flush=True,
        )
    # Unsloth's patched generate() does not accept the Transformers `generator` kwarg.
    # Reset all global RNGs per sample instead, preserving deterministic resume behavior.
    sample_seed = stable_seed(sample_id)
    random.seed(sample_seed)
    np.random.seed(sample_seed)
    torch.manual_seed(sample_seed)
    torch.cuda.manual_seed_all(sample_seed)
    started = time.time()
    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=effective_max_new_tokens,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            min_p=MIN_P,
            eos_token_id=generation_eos_token_ids,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = generated[0, input_tokens:]
    raw_generation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    thinking, final_answer, found_think_close = split_thinking(raw_generation)
    return {
        "sample_seed": sample_seed,
        "raw_generation": raw_generation,
        "thinking": thinking,
        "final_answer": final_answer,
        "found_think_close": found_think_close,
        "input_tokens": input_tokens,
        "output_tokens": int(new_tokens.numel()),
        "effective_max_new_tokens": effective_max_new_tokens,
        "generation_seconds": time.time() - started,
    }


In [ ]:
# Dataset loading and schema checks. IFBench/IFEval call their sole evaluation split 'train'.
# LiveCodeBench code_generation_lite has only the official test split.
from datasets import disable_progress_bars, load_dataset
disable_progress_bars()
datasets_to_run = {}
if RUN_MATH:
    datasets_to_run["aime25"] = limit_dataset(load_dataset("math-ai/aime25", split="test"), LIMIT_AIME25)
    datasets_to_run["aime26"] = limit_dataset(load_dataset("math-ai/aime26", split="test"), LIMIT_AIME26)
    for name in ("aime25", "aime26"):
        required = {"problem", "answer"}
        assert required.issubset(datasets_to_run[name].column_names), (name, datasets_to_run[name].column_names)
if RUN_IF:
    datasets_to_run["ifbench"] = limit_dataset(load_dataset("allenai/IFBench_test", split="train"), LIMIT_IFBENCH)
    datasets_to_run["ifeval"] = limit_dataset(load_dataset("google/IFEval", split="train"), LIMIT_IFEVAL)
    for name in ("ifbench", "ifeval"):
        required = {"key", "prompt", "instruction_id_list", "kwargs"}
        assert required.issubset(datasets_to_run[name].column_names), (name, datasets_to_run[name].column_names)
if RUN_LCB:
    lcb_repo_path = Path(LCB_REPO_DIR).expanduser().resolve()
    lcb_package_path = lcb_repo_path / "lcb_runner"
    if not lcb_package_path.is_dir():
        raise FileNotFoundError(
            f"LiveCodeBench source was not found at {lcb_package_path}. Clone the fixed official "
            "repository revision before running this notebook."
        )
    actual_lcb_revision = subprocess.check_output(
        ["git", "-C", str(lcb_repo_path), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual_lcb_revision != LCB_REPO_REVISION:
        raise RuntimeError(
            f"LiveCodeBench revision mismatch: expected {LCB_REPO_REVISION}, got {actual_lcb_revision}. "
            f"Run: git -C {lcb_repo_path} checkout {LCB_REPO_REVISION}"
        )
    if str(lcb_repo_path) not in sys.path:
        sys.path.insert(0, str(lcb_repo_path))
    from lcb_runner.benchmarks.code_generation import CodeGenerationProblem

    # Prefer each revision-pinned local shard and download only shards that are missing.
    lcb_data_dir = Path(LCB_DATA_DIR).expanduser().resolve()
    hf_endpoint = os.environ.get("HF_ENDPOINT", "https://huggingface.co").rstrip("/")
    lcb_parquet_files = []
    local_lcb_shards = 0
    for index in range(9):
        filename = f"test-{index:05d}-of-00009.parquet"
        local_path = lcb_data_dir / filename
        if local_path.is_file():
            lcb_parquet_files.append(str(local_path))
            local_lcb_shards += 1
        else:
            lcb_parquet_files.append(
                f"{hf_endpoint}/datasets/livecodebench/code_generation_lite/resolve/"
                f"{LCB_DATASET_REVISION}/{LCB_RELEASE_VERSION}/{filename}"
            )
    remote_lcb_shards = 9 - local_lcb_shards
    print(
        f"Loading LiveCodeBench {LCB_RELEASE_VERSION}: {local_lcb_shards}/9 local shards "
        f"from {lcb_data_dir}; {remote_lcb_shards}/9 missing shards will be downloaded.",
        flush=True,
    )
    raw_lcb_dataset = load_dataset(
        "parquet", data_files={"test": lcb_parquet_files}, split="test"
    )
    lcb_dataset = [CodeGenerationProblem(**row) for row in raw_lcb_dataset]
    lcb_loader_mode = (
        "local_pinned_parquet_shards" if remote_lcb_shards == 0
        else "local_first_remote_fallback"
    )
    lcb_dataset = sorted(lcb_dataset, key=lambda problem: str(problem.question_id))
    if LIMIT_LCB is not None:
        lcb_dataset = lcb_dataset[:min(int(LIMIT_LCB), len(lcb_dataset))]
    datasets_to_run["livecodebench"] = lcb_dataset

dataset_audit = {}
for name, dataset in datasets_to_run.items():
    if name == "livecodebench":
        dataset_audit[name] = {
            "rows": len(dataset), "dataset": "livecodebench/code_generation_lite",
            "release": LCB_RELEASE_VERSION, "split": "test", "has_train_split": False,
            "repository_revision": actual_lcb_revision,
            "dataset_revision": LCB_DATASET_REVISION, "loader_mode": lcb_loader_mode,
            "local_data_dir": str(lcb_data_dir), "local_shards": local_lcb_shards,
            "remote_shards": remote_lcb_shards,
        }
    else:
        dataset_audit[name] = {"rows": len(dataset), "columns": dataset.column_names}
(EVAL_OUTPUT_DIR / "dataset_audit.json").write_text(
    json.dumps(dataset_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(dataset_audit, ensure_ascii=False, indent=2), flush=True)


## Math: AIME25 and AIME26

The primary metric is exact match after extracting the last `\boxed{...}` (with conservative fallbacks) and normalizing the AIME integer answer.


In [ ]:
def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", text)]
    for start in reversed(starts):
        brace_start = text.find("{", start)
        depth = 0
        for index in range(brace_start, len(text)):
            if text[index] == "{":
                depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0:
                    return text[brace_start + 1:index]
    return None

def extract_math_answer(text):
    boxed = extract_last_boxed(text)
    if boxed is not None:
        return boxed
    final_matches = re.findall(r"(?i)final\s+answer\s*(?:is|:|=)?\s*([^\n]+)", text)
    if final_matches:
        return final_matches[-1].strip()
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", text.replace(",", ""))
    return integers[-1] if integers else ""

def normalize_math_answer(value):
    value = str(value).strip()
    value = value.replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value):
        value = str(int(value))
    return value

def run_math_benchmark(name, dataset):
    output_path = RAW_DIR / f"{name}.jsonl"
    prior = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(example.get("id", index)) for index, example in enumerate(dataset)}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] fed-mopsd-global/{name}: all {len(dataset)} samples already completed", flush=True)
        return prior
    started = time.time()
    if completed:
        print(f"[RESUME] fed-mopsd-global/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"fed-mopsd-global/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for index, example in enumerate(dataset):
        sample_id = str(example.get("id", index))
        if sample_id in completed:
            done += 1
            continue
        prompt = (
            str(example["problem"]).strip()
            + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
        )
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "problem": example["problem"],
            "gold_answer": str(example["answer"]),
        }
        try:
            result = generate_one(prompt, MAX_NEW_TOKENS, f"{name}:{sample_id}")
            predicted = extract_math_answer(result["final_answer"])
            record = {
                **base, **result, "status": "ok",
                "predicted_answer": predicted,
                "correct": normalize_math_answer(predicted) == normalize_math_answer(example["answer"]),
            }
            consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "correct": False}
            consecutive_errors += 1
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"fed-mopsd-global/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(f"Aborting {name} after {consecutive_errors} consecutive generation errors; last error: {record.get('error')}")
    return load_matching_records(output_path)

math_records = {}
if RUN_MATH:
    math_records["aime25"] = run_math_benchmark("aime25", datasets_to_run["aime25"])
    math_records["aime26"] = run_math_benchmark("aime26", datasets_to_run["aime26"])


## Instruction following: IFBench and IFEval

The code below mirrors AllenAI's official strict/loose verifier logic. Crucially, only text after the model's final `</think>` is checked; hidden reasoning must not contaminate length, keyword, or formatting constraints. Primary reported scores are prompt-level loose accuracy, alongside all four official aggregate variants.


In [ ]:
# IFBench's official verifiers require these NLTK resources. Download them before
# importing ifbench so a network problem fails before any new model generations.
import nltk

NLTK_DATA_DIR = Path(NLTK_DATA_DIR).expanduser().resolve()
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
os.environ["NLTK_DATA"] = str(NLTK_DATA_DIR)
if str(NLTK_DATA_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR))
NLTK_RESOURCES = {} if LCB_EVAL_ONLY else {
    "punkt_tab": "tokenizers/punkt_tab",
    "stopwords": "corpora/stopwords",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
}
for package_name, resource_path in NLTK_RESOURCES.items():
    try:
        nltk.data.find(resource_path)
        print(f"[NLTK] Found {package_name}", flush=True)
    except LookupError:
        print(f"[NLTK] Downloading {package_name} to {NLTK_DATA_DIR}", flush=True)
        last_error = None
        for attempt in range(1, 4):
            try:
                downloaded = nltk.download(
                    package_name, download_dir=str(NLTK_DATA_DIR), quiet=True, raise_on_error=True
                )
                if not downloaded:
                    raise RuntimeError("nltk.download returned False")
                nltk.data.find(resource_path)
                last_error = None
                break
            except Exception as exc:
                last_error = exc
                print(f"[NLTK] {package_name} attempt {attempt}/3 failed: {exc!r}", flush=True)
                if attempt < 3:
                    time.sleep(5 * attempt)
        if last_error is not None:
            raise RuntimeError(
                f"Unable to install required NLTK resource {package_name!r}. "
                f"Run `python -m nltk.downloader -d {NLTK_DATA_DIR} {package_name}` and retry."
            ) from last_error

instructions_registry = None
if not LCB_EVAL_ONLY:
    from ifbench import instructions_registry

def response_variants(response):
    lines = response.split("\n")
    remove_first = "\n".join(lines[1:]).strip()
    remove_last = "\n".join(lines[:-1]).strip()
    remove_both = "\n".join(lines[1:-1]).strip()
    return [
        response,
        response.replace("*", ""),
        remove_first,
        remove_last,
        remove_both,
        remove_first.replace("*", ""),
        remove_last.replace("*", ""),
        remove_both.replace("*", ""),
    ]

def check_instruction(instruction_id, kwargs, prompt, candidates):
    instruction_cls = instructions_registry.INSTRUCTION_DICT[instruction_id]
    instruction = instruction_cls(instruction_id)
    clean_kwargs = {key: value for key, value in dict(kwargs).items() if value is not None}
    instruction.build_description(**clean_kwargs)
    args = instruction.get_instruction_args()
    if args and "prompt" in args:
        instruction.build_description(prompt=prompt)
    return any(candidate.strip() and instruction.check_following(candidate) for candidate in candidates)

def score_instruction_response(example, response):
    strict = []
    loose = []
    for instruction_id, kwargs in zip(example["instruction_id_list"], example["kwargs"]):
        strict.append(check_instruction(instruction_id, kwargs, example["prompt"], [response]))
        loose.append(check_instruction(instruction_id, kwargs, example["prompt"], response_variants(response)))
    return strict, loose

def run_if_benchmark(name, dataset):
    output_path = RAW_DIR / f"{name}.jsonl"
    prior = load_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(example.get("key", index)) for index, example in enumerate(dataset)}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] fed-mopsd-global/{name}: all {len(dataset)} samples already completed", flush=True)
        return prior
    started = time.time()
    if completed:
        print(f"[RESUME] fed-mopsd-global/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"fed-mopsd-global/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for index, example in enumerate(dataset):
        sample_id = str(example.get("key", index))
        if sample_id in completed:
            done += 1
            continue
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "prompt": example["prompt"],
            "instruction_id_list": example["instruction_id_list"],
            "kwargs": example["kwargs"],
        }
        try:
            result = generate_one(example["prompt"], MAX_NEW_TOKENS, f"{name}:{sample_id}")
            strict, loose = score_instruction_response(example, result["final_answer"])
            record = {
                **base, **result, "status": "ok",
                "strict_instruction_results": strict,
                "loose_instruction_results": loose,
                "strict_prompt_pass": all(strict),
                "loose_prompt_pass": all(loose),
            }
            consecutive_errors = 0
        except Exception as exc:
            consecutive_errors += 1
            failed = [False] * len(example["instruction_id_list"])
            record = {
                **base, "status": "error", "error": repr(exc),
                "strict_instruction_results": failed,
                "loose_instruction_results": failed,
                "strict_prompt_pass": False, "loose_prompt_pass": False,
            }
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"fed-mopsd-global/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(f"Aborting {name} after {consecutive_errors} consecutive generation/scoring errors; last error: {record.get('error')}")
    return load_matching_records(output_path)

if_records = {}
if RUN_IF:
    if_records["ifbench"] = run_if_benchmark("ifbench", datasets_to_run["ifbench"])
    if_records["ifeval"] = run_if_benchmark("ifeval", datasets_to_run["ifeval"])


## LiveCodeBench code generation

Generation uses the official generic code-generation instructions plus the checkpoint's verified Qwen ChatML template. The official local Python checker computes pass@1 over the saved programs. Generation is resumable per problem, and `LCB_EVAL_ONLY=true` reruns only the checker.


In [ ]:
# Import the prompt helpers from the fixed official source tree. This revision opens
# its few-shot assets relative to the repository root during import.
# Import only the pinned code-generation prompt module. Importing it normally first
# executes lcb_runner.prompts.__init__, which eagerly imports optional Anthropic-only
# prompt modules even though this local Qwen evaluation does not use Anthropic.
import importlib.util
_notebook_cwd = Path.cwd()
_prompt_module_path = lcb_repo_path / "lcb_runner" / "prompts" / "code_generation.py"
_prompt_spec = importlib.util.spec_from_file_location(
    "_pinned_lcb_code_generation_prompt", _prompt_module_path
)
if _prompt_spec is None or _prompt_spec.loader is None:
    raise ImportError(f"Unable to load LiveCodeBench prompt module: {_prompt_module_path}")
_prompt_module = importlib.util.module_from_spec(_prompt_spec)
try:
    os.chdir(lcb_repo_path)
    _prompt_spec.loader.exec_module(_prompt_module)
finally:
    os.chdir(_notebook_cwd)
PromptConstants = _prompt_module.PromptConstants
get_generic_question_template_answer = _prompt_module.get_generic_question_template_answer
from lcb_runner.lm_styles import LMStyle
from lcb_runner.utils.extraction_utils import extract_code

LCB_PROMPT_VERSION = "official-generic-chat-v1"
lcb_fingerprint_payload = {
    "run_fingerprint": RUN_FINGERPRINT,
    "repository_revision": LCB_REPO_REVISION,
    "dataset_revision": LCB_DATASET_REVISION,
    "release_version": LCB_RELEASE_VERSION,
    "prompt_version": LCB_PROMPT_VERSION,
}
LCB_RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(lcb_fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"LiveCodeBench run fingerprint: {LCB_RUN_FINGERPRINT}", flush=True)

def load_lcb_matching_records(path):
    records = load_matching_records(path)
    return {
        sample_id: record for sample_id, record in records.items()
        if record.get("lcb_run_fingerprint") == LCB_RUN_FINGERPRINT
    }

def make_lcb_prompt(problem):
    # This is LiveCodeBench's official generic user prompt, including its required
    # fenced-Python output format. The official system instruction is passed separately.
    return get_generic_question_template_answer(problem)

def run_lcb_generation(dataset):
    name = "livecodebench"
    output_path = RAW_DIR / "livecodebench.jsonl"
    prior = load_lcb_matching_records(output_path) if RESUME else {}
    completed = {key for key, value in prior.items() if value.get("status") == "ok"}
    expected_ids = {str(problem.question_id) for problem in dataset}
    if RESUME and expected_ids.issubset(completed):
        print(f"[SKIP] fed-mopsd-global/{name}: all {len(dataset)} samples already completed", flush=True)
        return prior
    started = time.time()
    if completed:
        print(f"[RESUME] fed-mopsd-global/{name}: {len(completed)} completed matching records", flush=True)
        print_progress(f"fed-mopsd-global/{name}", len(completed), len(dataset), started)
    done = 0
    consecutive_errors = 0
    for problem in dataset:
        sample_id = str(problem.question_id)
        if sample_id in completed:
            done += 1
            continue
        base = {
            "run_fingerprint": RUN_FINGERPRINT,
            "lcb_run_fingerprint": LCB_RUN_FINGERPRINT,
            "benchmark": name,
            "sample_id": sample_id,
            "question_title": problem.question_title,
            "question_content": problem.question_content,
            "platform": problem.platform.value,
            "contest_id": problem.contest_id,
            "contest_date": problem.contest_date.isoformat(),
            "difficulty": problem.difficulty.value,
            "starter_code": problem.starter_code,
        }
        try:
            result = generate_one(
                make_lcb_prompt(problem), MAX_NEW_TOKENS, f"{name}:{sample_id}",
                system_prompt=PromptConstants.SYSTEM_MESSAGE_GENERIC,
            )
            # The official prompt requires fenced code. LiveCodeBench's own extractor
            # returns the last fenced block and returns an empty program if no block exists.
            extracted_code = extract_code(result["final_answer"], LMStyle.CodeQwenInstruct)
            record = {**base, **result, "status": "ok", "extracted_code": extracted_code}
            consecutive_errors = 0
        except Exception as exc:
            record = {**base, "status": "error", "error": repr(exc), "extracted_code": ""}
            consecutive_errors += 1
            print(f"[ERROR] {name}:{sample_id}: {exc!r}", flush=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        append_jsonl(output_path, record)
        prior[sample_id] = record
        done += 1
        print_progress(f"fed-mopsd-global/{name}", done, len(dataset), started)
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            raise RuntimeError(
                f"Aborting {name} after {consecutive_errors} consecutive generation errors; "
                f"last error: {record.get('error')}"
            )
    return load_lcb_matching_records(output_path)

lcb_records = {}
if RUN_LCB:
    if LCB_EVAL_ONLY:
        lcb_records = load_lcb_matching_records(RAW_DIR / "livecodebench.jsonl")
        print(f"[SKIP] Generation disabled; loaded {len(lcb_records)} matching LiveCodeBench records.", flush=True)
    else:
        lcb_records = run_lcb_generation(datasets_to_run["livecodebench"])


In [ ]:
# Run the fixed-revision official checker while emitting one durable log line per
# completed problem instead of a terminal-rewriting progress bar.
lcb_eval_path = EVAL_OUTPUT_DIR / "livecodebench_eval.json"
lcb_eval_status_path = EVAL_OUTPUT_DIR / "livecodebench_eval_status.json"
lcb_evaluation = {"status": "not_run", "correct": None, "total": None, "pass@1": None}

class LineProgress:
    def __init__(self, *args, total=None, **kwargs):
        self.total = int(total or 0)
        self.done = 0
        self.started = time.time()
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, traceback):
        return False
    def update(self, amount=1):
        self.done += int(amount)
        print_progress("fed-mopsd-global/livecodebench-check", self.done, self.total, self.started)

if RUN_LCB and RUN_LCB_EVALUATION:
    lcb_dataset = datasets_to_run["livecodebench"]
    expected_ids = [str(problem.question_id) for problem in lcb_dataset]
    missing_ids = [
        sample_id for sample_id in expected_ids
        if lcb_records.get(sample_id, {}).get("status") != "ok"
    ]
    if missing_ids:
        raise RuntimeError(
            f"LiveCodeBench evaluation requires all generations: {len(missing_ids)} of "
            f"{len(expected_ids)} are missing or failed. Rerun generation first. "
            f"First missing IDs: {missing_ids[:5]}"
        )
    expected_ids_hash = hashlib.sha256(
        json.dumps(expected_ids, ensure_ascii=False).encode("utf-8")
    ).hexdigest()
    lcb_eval_fingerprint = hashlib.sha256(json.dumps({
        "lcb_run_fingerprint": LCB_RUN_FINGERPRINT,
        "expected_ids_hash": expected_ids_hash,
        "timeout": LCB_TIMEOUT_SECONDS,
        "repository_revision": LCB_REPO_REVISION,
        "dataset_revision": LCB_DATASET_REVISION,
    }, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    previous_status = {}
    if RESUME and lcb_eval_status_path.is_file() and lcb_eval_path.is_file():
        try:
            previous_status = json.loads(lcb_eval_status_path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"[WARN] Ignoring unreadable LiveCodeBench evaluation status: {exc!r}", flush=True)
    if (previous_status.get("status") == "complete"
            and previous_status.get("eval_fingerprint") == lcb_eval_fingerprint):
        lcb_evaluation = previous_status
        print(
            f"[SKIP] LiveCodeBench checker already completed: "
            f"{lcb_evaluation['correct']}/{lcb_evaluation['total']} "
            f"(pass@1={100 * lcb_evaluation['pass@1']:.2f}%).",
            flush=True,
        )
    else:
        import lcb_runner.evaluation.compute_code_generation_metrics as lcb_metric_module
        from lcb_runner.evaluation.pass_k_utils import extract_instance_results
        lcb_metric_module.tqdm = LineProgress
        samples_list = [problem.get_evaluation_sample() for problem in lcb_dataset]
        generations_list = [[lcb_records[str(problem.question_id)]["extracted_code"]] for problem in lcb_dataset]
        official_metrics, official_results, official_metadata = lcb_metric_module.codegen_metrics(
            samples_list,
            generations_list,
            k_list=[1],
            num_process_evaluate=int(LCB_NUM_PROCESSES),
            timeout=int(LCB_TIMEOUT_SECONDS),
            debug=False,
        )
        instance_grades = extract_instance_results(official_results)
        graded = [bool(grades[0]) for grades in instance_grades]
        correct = sum(graded)
        total = len(graded)
        pass_at_1 = float(official_metrics["pass@1"])
        assert math.isclose(pass_at_1, correct / total, rel_tol=0.0, abs_tol=1e-12)
        per_problem = [
            {
                "question_id": expected_ids[index],
                "passed": graded[index],
                "test_results": official_results[index][0],
                "metadata": official_metadata[index][0],
            }
            for index in range(total)
        ]
        eval_payload = {
            "model": MODEL_NAME,
            "release_version": LCB_RELEASE_VERSION,
            "repository_revision": LCB_REPO_REVISION,
            "dataset_revision": LCB_DATASET_REVISION,
            "lcb_run_fingerprint": LCB_RUN_FINGERPRINT,
            "eval_fingerprint": lcb_eval_fingerprint,
            "official_metrics": official_metrics,
            "correct": correct,
            "total": total,
            "pass@1": pass_at_1,
            "per_problem": per_problem,
        }
        lcb_eval_path.write_text(json.dumps(eval_payload, ensure_ascii=False, indent=2), encoding="utf-8")
        lcb_evaluation = {
            "status": "complete",
            "correct": correct,
            "total": total,
            "pass@1": pass_at_1,
            "eval_fingerprint": lcb_eval_fingerprint,
            "result_path": str(lcb_eval_path),
        }
        lcb_eval_status_path.write_text(
            json.dumps(lcb_evaluation, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print(f"[RESULT] LiveCodeBench pass@1: {correct}/{total} = {100 * pass_at_1:.2f}%", flush=True)
elif RUN_LCB:
    print("[SKIP] RUN_LCB_EVALUATION=False; generations were saved but not scored.", flush=True)


## Aggregate metrics and bar chart


In [ ]:
metric_rows = []
if LCB_EVAL_ONLY:
    prior_metrics_path = EVAL_OUTPUT_DIR / "metrics.json"
    if prior_metrics_path.is_file():
        prior_payload = json.loads(prior_metrics_path.read_text(encoding="utf-8"))
        metric_rows = [
            row for row in prior_payload.get("metrics", [])
            if row.get("benchmark") != "LiveCodeBench"
        ]
        print(f"[RESUME] Preserved {len(metric_rows)} completed non-LiveCodeBench metric rows.", flush=True)
    else:
        print("[WARN] No prior metrics.json found; evaluation-only output will contain LiveCodeBench only.", flush=True)

def add_metric(benchmark, domain, metric, numerator, denominator, status="complete", note=""):
    score = numerator / denominator if numerator is not None and denominator else None
    metric_rows.append({
        "benchmark": benchmark, "domain": domain, "metric": metric,
        "score": score, "score_percent": 100 * score if score is not None else None,
        "numerator": numerator, "denominator": denominator, "status": status, "note": note,
    })

for name, records_by_id in math_records.items():
    selected_ids = [str(row.get("id", index)) for index, row in enumerate(datasets_to_run[name])]
    records = [records_by_id.get(sample_id, {}) for sample_id in selected_ids]
    add_metric(name.upper(), "Math", "exact_match", sum(bool(record.get("correct")) for record in records), len(records))

for name, records_by_id in if_records.items():
    selected_ids = [str(row.get("key", index)) for index, row in enumerate(datasets_to_run[name])]
    records = [records_by_id.get(sample_id, {}) for sample_id in selected_ids]
    strict_lists = [record.get("strict_instruction_results", []) for record in records]
    loose_lists = [record.get("loose_instruction_results", []) for record in records]
    instruction_total = sum(len(values) for values in strict_lists)
    add_metric(name.upper(), "Instruction Following", "prompt_level_strict", sum(bool(values) and all(values) for values in strict_lists), len(records))
    add_metric(name.upper(), "Instruction Following", "instruction_level_strict", sum(sum(values) for values in strict_lists), instruction_total)
    add_metric(name.upper(), "Instruction Following", "prompt_level_loose", sum(bool(values) and all(values) for values in loose_lists), len(records))
    add_metric(name.upper(), "Instruction Following", "instruction_level_loose", sum(sum(values) for values in loose_lists), instruction_total)

if RUN_LCB:
    if lcb_evaluation.get("status") == "complete":
        add_metric("LiveCodeBench", "Code", "pass@1", lcb_evaluation["correct"], lcb_evaluation["total"])
    else:
        metric_rows.append({
            "benchmark": "LiveCodeBench", "domain": "Code", "metric": "pass@1",
            "score": None, "score_percent": None, "numerator": None,
            "denominator": lcb_evaluation.get("total"), "status": "not_evaluated",
            "note": "The official LiveCodeBench checker was not run.",
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_csv = EVAL_OUTPUT_DIR / "metrics.csv"
metrics_json = EVAL_OUTPUT_DIR / "metrics.json"
metrics_df.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
metrics_payload = {
    "model": EVAL_MODEL_NAME,
    "base_model": MODEL_NAME,
    "training_fingerprint": TRAINING_FINGERPRINT, "run_fingerprint": RUN_FINGERPRINT,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "metrics": metric_rows, "livecodebench_evaluation": lcb_evaluation,
}
metrics_json.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding="utf-8")
display(metrics_df)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

primary_specs = [
    ("AIME25", "exact_match", "AIME25\nExact match"),
    ("AIME26", "exact_match", "AIME26\nExact match"),
    ("IFBENCH", "prompt_level_loose", "IFBench\nPrompt loose"),
    ("IFEVAL", "prompt_level_loose", "IFEval\nPrompt loose"),
    ("LiveCodeBench", "pass@1", "LiveCodeBench\nPass@1"),
]
labels, values, available = [], [], []
for benchmark, metric, label in primary_specs:
    row = metrics_df[(metrics_df["benchmark"] == benchmark) & (metrics_df["metric"] == metric)]
    if row.empty:
        continue
    value = row.iloc[0]["score_percent"]
    labels.append(label)
    available.append(pd.notna(value))
    values.append(float(value) if pd.notna(value) else 0.0)

colors = ["#4C78A8", "#4C78A8", "#F58518", "#F58518", "#54A24B"][:len(labels)]
fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="#333333", linewidth=0.7)
for bar, value, is_available in zip(bars, values, available):
    if is_available:
        ax.text(bar.get_x() + bar.get_width() / 2, value + 1.2, f"{value:.2f}%", ha="center", va="bottom", fontsize=10)
    else:
        bar.set_facecolor("#D9D9D9")
        bar.set_hatch("//")
        ax.text(bar.get_x() + bar.get_width() / 2, 2.0, "N/A\n(not evaluated)", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(len(labels)), labels)
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 108)
ax.set_title("Fed-MOPSD Qwen3-4B: Aggregated Student Evaluation")
ax.grid(axis="y", linestyle="--", alpha=0.3)
fig.tight_layout()
figure_path = EVAL_OUTPUT_DIR / "fed_mopsd_global_scores.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Metrics JSON: {metrics_json}", flush=True)
print(f"Metrics CSV:  {metrics_csv}", flush=True)
print(f"Bar chart:    {figure_path}", flush=True)


In [ ]:
gc.collect()
if not LCB_EVAL_ONLY and torch.cuda.is_available():
    torch.cuda.empty_cache()
print("[DONE] Fed-MOPSD training, FedAvg, evaluation, and chart generation completed.", flush=True)
